# DriveSense-VLM — 01: SFT Training

**GPU**: A100 80GB (required) | **Time**: ~5–7 h | **Cost**: ~72 CU

Fine-tunes Qwen3-VL-2B-Instruct with LoRA (rank 32, α 64) on DriveSense data.

> ⚠️ **Before running**: Runtime → Change runtime type → **A100 GPU**
>
> **RECOVERY**: If Colab disconnects mid-training, rerun Cells 2–3 (setup + install),
> then rerun the training cell. It auto-resumes from the latest Drive checkpoint.

**Prerequisites**: Run `00_data_pipeline.ipynb` first to generate `sft_train.jsonl`.

In [ ]:
# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ── Paths ──
PROJECT_ROOT = "/content/drive/MyDrive/DriveSense-VLM"
REPO_ROOT    = "/content/DriveSense-VLM"
DATA_ROOT    = f"{PROJECT_ROOT}/data"
OUTPUTS_ROOT = f"{PROJECT_ROOT}/outputs"

# Create Drive directories
for d in [DATA_ROOT, f"{DATA_ROOT}/nuscenes", f"{DATA_ROOT}/dada2000",
          OUTPUTS_ROOT, f"{OUTPUTS_ROOT}/data", f"{OUTPUTS_ROOT}/training",
          f"{OUTPUTS_ROOT}/data/sft_ready"]:
    os.makedirs(d, exist_ok=True)

# Clone repo to fast ephemeral SSD
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/jayanth922/DriveSense-VLM.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
os.chdir(REPO_ROOT)

# Symlink data/outputs → Drive (persistent across disconnects)
!ln -sfn {DATA_ROOT} {REPO_ROOT}/data
!ln -sfn {OUTPUTS_ROOT} {REPO_ROOT}/outputs

# Add src to Python path (replaces broken editable install)
sys.path.insert(0, f"{REPO_ROOT}/src")
print(f"✓ Repo:  {REPO_ROOT}")
print(f"✓ Drive: {PROJECT_ROOT}")

In [ ]:
# Note: nuscenes-devkit (used in nb00/05) must be installed with --no-deps
#       to skip its strict numpy version pin. Not needed in this notebook.
# Install training dependencies
!pip install --upgrade pip setuptools wheel -q 2>&1 | tail -1
!pip install pyyaml tqdm Pillow requests -q 2>&1 | tail -1
!pip install transformers peft accelerate datasets bitsandbytes wandb -q 2>&1 | tail -1
!pip install flash-attn --no-build-isolation -q 2>&1 | tail -3

# Verify GPU
import torch
assert torch.cuda.is_available(), (
    "No GPU detected!\n"
    "Fix: Runtime → Change runtime type → A100 GPU"
)
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✓ GPU : {gpu}")
print(f"✓ VRAM: {vram:.1f} GB")
if "A100" not in gpu:
    print(f"  WARNING: Expected A100 — got {gpu}. May OOM on <40 GB GPUs.")

import drivesense
print(f"✓ drivesense importable")

In [ ]:
# Check for existing checkpoints (resume support)
import os, glob

TRAIN_OUT = f"{OUTPUTS_ROOT}/training"
os.makedirs(TRAIN_OUT, exist_ok=True)

checkpoints = sorted(glob.glob(f"{TRAIN_OUT}/checkpoint-*"))
if checkpoints:
    print(f"Found {len(checkpoints)} existing checkpoint(s) on Drive:")
    for c in checkpoints:
        print(f"  {c}")
    print(f"\nLatest: {checkpoints[-1]}")
    print("Training will AUTO-RESUME from the latest checkpoint.")
else:
    print("No checkpoints found — starting fresh training.")

# Verify SFT data exists
SFT_DIR = f"{OUTPUTS_ROOT}/data/sft_ready"
train_path = f"{SFT_DIR}/sft_train.jsonl"
if os.path.exists(train_path):
    with open(train_path) as f:
        n = sum(1 for _ in f)
    print(f"\n✓ Training data: {n} examples at {train_path}")
else:
    raise FileNotFoundError(
        f"sft_train.jsonl not found at {train_path}\n"
        "Run 00_data_pipeline.ipynb first."
    )

In [ ]:
# Configure Weights & Biases
import wandb
wandb.login()  # Prompts for API key — get it at https://wandb.ai/authorize
print("✓ W&B configured — metrics will log to project: drivesense-vlm")

## Training

Checkpoints saved to Google Drive every 250 steps (persistent across disconnects).
To resume after disconnect: rerun Cells 2–3, then rerun this cell.

In [ ]:
import os

SFT_DIR  = f"{OUTPUTS_ROOT}/data/sft_ready"
TRAIN_OUT = f"{OUTPUTS_ROOT}/training"

# Copy SFT data to fast ephemeral local SSD for faster DataLoader I/O
print("Copying SFT data to local /content/ for fast I/O...")
!cp {SFT_DIR}/sft_train.jsonl /content/sft_train.jsonl
!cp {SFT_DIR}/sft_val.jsonl   /content/sft_val.jsonl
print("✓ SFT data on fast local storage")

# Run training — outputs go to Drive (persistent)
# RECOVERY: If session disconnects, rerun cells 2-3 then rerun this cell.
#           Training auto-detects the latest checkpoint and resumes.
!python scripts/run_training.py \
    --config configs/training.yaml \
    --override training.output_dir={TRAIN_OUT} \
    --override training.save_steps=250 \
    --override training.save_total_limit=3 \
    --resume \
    2>&1

## Results

In [ ]:
import os, json, glob

TRAIN_OUT = f"{OUTPUTS_ROOT}/training"

# Display training metrics
metrics_path = f"{TRAIN_OUT}/training_metrics.json"
if os.path.exists(metrics_path):
    m = json.loads(open(metrics_path).read())
    print("Training Metrics:")
    print("─" * 40)
    for k, v in m.items():
        print(f"  {k:<30}: {v}")
else:
    print(f"Metrics file not found at {metrics_path}")

# Verify best checkpoint saved
best = f"{TRAIN_OUT}/checkpoint-best"
if not os.path.exists(best):
    ckpts = sorted(glob.glob(f"{TRAIN_OUT}/checkpoint-*"))
    best = ckpts[-1] if ckpts else None

if best and os.path.exists(best):
    print(f"\n✓ Adapter saved: {best}")
    !ls -lh {best}
    print("\nProceed to 02_optimization.ipynb")
else:
    print("⚠ No checkpoint found — training may still be running or failed.")